# Hitting-Time Expectation Generator

Notebook workbench for generating prompts, reasoning traces, and canonical answers for hitting-time expectation problems.

In [1]:
from pathlib import Path
import json
import sys

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "benchmark").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
from benchmark.generators import HittingTimeExpectationGenerator

gen = HittingTimeExpectationGenerator()

In [3]:
params = gen.sample_params(seed=401, difficulty=2, split="dev")
params

{'problem_type': 'symmetric_shifted_boundaries',
 'difficulty': 2,
 'split': 'dev',
 'seed': 401,
 'process_notation': 'X',
 'lower': -5,
 'upper': 3,
 'start': 2,
 'expected_time': Fraction(7, 1)}

In [4]:
problem = gen.generate_problem(params)
reasoning = gen.generate_reasoning(params)
solution = gen.generate_solution(params)

print("PROBLEM:\n", problem)
print("\nREASONING:\n", reasoning)
print("\nSOLUTION:\n", gen.to_json_safe(solution))

PROBLEM:
 Let (X_n) be a simple symmetric random walk with X_0 = 2. Let tau be the first time the walk hits either -5 or 3. Compute E[tau]. Answer with JSON of the form {"expected_time": "..."} inside the answer tags.

REASONING:
 Use the symmetric gambler's ruin hitting-time formula: for a simple symmetric random walk started at i and stopped when it first hits 0 or a, E_i[tau] = i(a-i). Reduce to the standard finite-boundary simple symmetric random walk. Shift the walk by subtracting the lower boundary -5. The shifted start is 7 and the shifted upper boundary is 8. The symmetric gambler's ruin hitting-time formula gives E[tau] = (start-lower)(upper-start).

Final answer:
<answer>
{"expected_time": "7"}
</answer>

SOLUTION:
 {'expected_time': '7'}


In [5]:
records = []
for difficulty in [1, 2, 3]:
    for seed in range(4000 + 100 * difficulty, 4005 + 100 * difficulty):
        records.append(gen.generate_record(seed=seed, difficulty=difficulty, split="dev"))

len(records), records[0]

(15,
 {'id': 'hitting_time_expectation_dev_004100',
  'family': 'hitting_time_expectation',
  'problem_type': 'symmetric_boundaries_zero_a',
  'difficulty': 1,
  'split': 'dev',
  'seed': 4100,
  'params': {'problem_type': 'symmetric_boundaries_zero_a',
   'difficulty': 1,
   'split': 'dev',
   'seed': 4100,
   'process_notation': 'X',
   'lower': 0,
   'upper': 10,
   'start': 3,
   'expected_time': '21'},
  'problem': 'Let (X_n) be a simple symmetric random walk on the integers with X_0 = 3. Let tau = inf{n >= 0 : X_n in {0, 10}}. Compute E[tau]. Answer with JSON of the form {"expected_time": "..."} inside the answer tags.',
  'reasoning': 'Use optional stopping for a bounded stopped process: if (M_n) is a martingale and the stopped family (M_{n wedge tau}) is bounded, then E[M_tau] = E[M_0] for the finite-valued limit. Use two martingales for the finite-boundary simple symmetric random walk. First, (X_n) is a martingale, and optional stopping gives E[X_tau] = 3. Use the quadratic ma

In [6]:
output_path = project_root / "benchmark" / "data" / "dev" / "hitting_time_expectation_preview.jsonl"
with output_path.open("w") as f:
    for record in records:
        f.write(json.dumps(record, sort_keys=True) + "\n")

output_path

PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/benchmark/data/dev/hitting_time_expectation_preview.jsonl')